> 本 Notebook 由对应 Word 实验手册生成。只有“测试与验收”章节中的测试指令可执行；其余代码仅用于阅读和讲解。


## 实验八：SwiGLU算子的优化与验证


建议学时：4学时


## 实验任务

1.任务描述


本实验在SwiGLU基础版上进行优化。优化版保持外部调用接口不变，把逐元素全局内存访问改为片上缓冲区分块搬运，并使用AscendC向量接口完成激活和乘法计算。


2.学习目标


完成本实验后，学生应能够：


• 理解SwiGLU优化版中向量化和流水组织的设计动机。


• 掌握TPipe、TQue、TBuf和LocalTensor的基本使用方法。


• 能够用DataCopy与向量API实现SwiGLU的分块计算。


• 能够从正确性、模型替换和性能结果三个角度分析优化效果。


## 任务准备


1.优化前的瓶颈与策略总览


SwiGLU基础版已经完成了公式验证和模型替换，但它采用逐元素GM读写和标量计算，主要用于建立正确性基线。优化版在不改变公式和PyTorch接口的前提下，调整数据流和计算粒度，把输入分块搬入UB后使用向量接口完成计算。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">优化前瓶颈</th>
<th style="text-align:left;">影响</th>
<th style="text-align:left;">优化策略</th>
<th style="text-align:left;">代码应用</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">逐元素访问GM</td>
<td style="text-align:left;">访存粒度小，读写次数多</td>
<td style="text-align:left;">UB分块搬运</td>
<td style="text-align:left;">使用DataCopy按tile搬入gate/up</td>
</tr>
<tr>
<td style="text-align:left;">标量计算粒度小</td>
<td style="text-align:left;">每次只处理一个元素</td>
<td style="text-align:left;">向量化计算</td>
<td style="text-align:left;">使用Muls/Exp/Adds/Div/Mul批量处理</td>
</tr>
<tr>
<td style="text-align:left;">搬运与计算边界不清晰</td>
<td style="text-align:left;">数据生命周期不够明确</td>
<td style="text-align:left;">流水组织</td>
<td style="text-align:left;">使用TPipe/TQue/TBuf组织CopyIn、Compute、CopyOut</td>
</tr>
<tr>
<td style="text-align:left;">尾块长度不固定</td>
<td style="text-align:left;">输入规模不一定被tile整除</td>
<td style="text-align:left;">尾块控制</td>
<td style="text-align:left;">使用curLength控制真实处理长度</td>
</tr>
</tbody></table>


优化版的核心目标是减少小粒度全局内存访问，并把计算放到片上缓冲区内批量完成。对SwiGLU来说，输入是一维连续数据，天然适合按tile切分；每个tile完成搬入、计算和写回三个阶段。优化版不改变SwiGLU数学语义，仍使用如下公式。


本次实验主要采用向量化、流水化两类核心优化手段。向量化的核心思路是将多组连续数据打包批量处理，依托硬件向量运算单元，在单次指令内完成一段数据的同类型运算。基础版本的 SwiGLU 算子逐元素调用 GetValue 读取数据、完成运算、再通过 SetValue 写回结果，单次仅能处理单个数据点位；优化实现会将 gate、up 的连续分块数据统一搬运至 UB 片上缓存，调用 Muls、Exp、Adds、Div、Mul 等硬件向量接口，一次性完成整个分块的批量运算。该方式能够大幅缩减标量循环的执行轮次，让激活函数、乘除等结构统一的运算适配硬件的批量并行特性。


流水化的设计思路是将完整算子执行流程拆解为多个串行阶段，让数据迁入、数值计算、结果回写形成可持续流转的数据流。本实验优化后的 SwiGLU 按照数据载入 CopyIn、运算计算 Compute、数据写出 CopyOut 三段流水线架构搭建：CopyIn 阶段依靠 DataCopy 接口，将 gate、up 数据从全局内存 GM 搬运至片上缓存 UB；Compute 阶段在 UB 内依次完成 sigmoid、silu 运算与最终相乘操作；CopyOut 阶段把运算结果写回全局内存 GM。代码依靠 TPipe 统一管控流水线硬件资源，TQue 承载输入、输出队列，TBuf 作为运算过程的临时缓存，LocalTensor 用于指代存放在 UB 内的本地张量。


两类优化在本算子内落地可归纳为三点设计逻辑。第一，借助 tileLength 对一维输入做定长分块，规避全量数据一次性运算造成的缓存资源超限问题。第二，通过 curLength 记录末尾不完整分块的有效数据长度，保证无法被分块尺寸整除的输入，依旧可以复用同一套运算流程。第三，将原本逐元素分步执行的数学公式拆解为批量向量运算，让每一个数据分块都能在 UB 中闭环完成全部 SwiGLU 运算。综上，优化方案在保证外部调用接口、数学计算公式完全不变的前提下，重构了数据流动路径与计算粒度，适配硬件并行特性。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">CopyIn：gate/up从全局内存搬入UB Compute：在UB中计算sigmoid、silu和最终乘法 CopyOut：把output从UB写回全局内存</th>
</tr>
</thead>
</table>


2.优化前后对比


下表从数据访问、计算方式、执行组织和模型测试结果四个角度对比基础版与优化版。运行结果来自“算子运行结果对比.doc”中的Qwen原生路径与自定义算子路径截图。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">对比项</th>
<th style="text-align:left;">基础版</th>
<th style="text-align:left;">优化版</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">数据访问</td>
<td style="text-align:left;">逐元素从GM读取并写回</td>
<td style="text-align:left;">按tile搬入UB后批量计算</td>
</tr>
<tr>
<td style="text-align:left;">计算方式</td>
<td style="text-align:left;">标量公式逐元素展开</td>
<td style="text-align:left;">向量接口分阶段完成整段数据计算</td>
</tr>
<tr>
<td style="text-align:left;">中间结果</td>
<td style="text-align:left;">主要保存在标量变量中</td>
<td style="text-align:left;">保存在LocalTensor和TBuf中</td>
</tr>
<tr>
<td style="text-align:left;">执行组织</td>
<td style="text-align:left;">单层元素循环</td>
<td style="text-align:left;">CopyIn、Compute、CopyOut三阶段</td>
</tr>
<tr>
<td style="text-align:left;">模型测试结果</td>
<td style="text-align:left;">custom路径约99.307 ms</td>
<td style="text-align:left;">custom路径约81.082 ms</td>
</tr>
</tbody></table>


3.算子定义与接口约定


优化版SwiGLU仍计算同一个门控激活公式，对外接口与基础版保持一致，因此测试脚本和Qwen替换逻辑可以复用。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">项目</th>
<th style="text-align:left;">工程约定</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">输入</td>
<td style="text-align:left;">gate和up为形状一致的float32类型NPU连续张量</td>
</tr>
<tr>
<td style="text-align:left;">输出</td>
<td style="text-align:left;">与输入形状一致</td>
</tr>
<tr>
<td style="text-align:left;">优化策略</td>
<td style="text-align:left;">UB分块搬运、向量化计算、CopyIn/Compute/CopyOut流水组织</td>
</tr>
<tr>
<td style="text-align:left;">核心参数</td>
<td style="text-align:left;">tileLength默认1024，尾块使用curLength处理</td>
</tr>
<tr>
<td style="text-align:left;">调用方式</td>
<td style="text-align:left;">torch.ops.swiglu_custom.swiglu(gate, up)</td>
</tr>
</tbody></table>


4.实验环境准备


本实验在Ascend NPU云服务器上完成，使用CANN工具链、AscendC和PyTorch NPU环境。基础版与优化版建议放在不同工程目录中，构建和测试也尽量在新的Python进程中执行，避免torch.library重复注册。


进入工程目录后，先加载CANN环境变量，再检查环境、构建工程并设置动态库搜索路径。CANN安装路径以云服务器实际配置为准。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">cd /home/user/SwiGluOptimizedExperiment source /usr/local/Ascend/ascend-toolkit/set_env.sh bash scripts/check_env.sh bash scripts/build.sh export LD_LIBRARY_PATH=$PWD/out/lib:$LD_LIBRARY_PATH</th>
</tr>
</thead>
</table>


## 任务实施


## 步骤一：明确优化目标


基础版的逐元素GM访问虽然简单，但访存粒度太细；优化版先搬入tile，再在UB里批量完成向量计算，最后一次性写回。这种改变让数据流更贴近硬件执行方式。


基础版每个元素都从全局内存读取、计算、写回，访存粒度小，计算接口也没有利用向量单元。优化版把一段连续元素作为tile搬入片上缓冲区，在本地张量上批量执行Exp、Div、Mul等向量计算，再把结果一次写回。这样可以减少反复访问全局内存的开销，并让计算更适合向量流水。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">基础版：全局内存 -&gt; 单元素读取 -&gt; 标量计算 -&gt; 单元素写回 优化版：全局内存 -&gt; DataCopy搬入UB -&gt; 向量计算 -&gt; DataCopy写回全局内存</th>
</tr>
</thead>
</table>


## 步骤二：扩展tiling信息


本步骤在基础版只记录总规模和算核分配的基础上，进一步加入tileLength。它的作用是描述每次搬运和计算的粒度，而curLength则描述尾块的真实长度。通过这两个字段，优化版可以把输入完整切分并逐段处理。


优化版除了总元素数和算核划分，还需要tileLength描述每次搬入片上缓冲区的元素个数。尾块长度由curLength动态计算，避免最后一段越界。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">struct SwiGluTilingData { uint32_t totalSize = 0; uint32_t elementsPerCore = 0; uint32_t coreNum = 1; uint32_t tileLength = 1024; };</th>
</tr>
</thead>
</table>


注册侧根据总元素个数选择实际启用的算核数量。小输入不强行启动8个算核，可以避免部分算核没有工作量时带来的调度浪费。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">uint32_t SelectBlockDim(uint32_t totalSize) { uint32_t blockDim = totalSize / kMinElementsPerCore; if (blockDim == 0) { blockDim = 1; } return std::min(kDefaultBlockDim, blockDim); } SwiGluTilingData BuildTiling(uint32_t totalSize, uint32_t blockDim) { SwiGluTilingData tiling {}; tiling.totalSize = totalSize; tiling.coreNum = blockDim; tiling.elementsPerCore = (totalSize + blockDim - 1U) / blockDim; tiling.tileLength = kDefaultTileLength; return tiling; }</th>
</tr>
</thead>
</table>


## 步骤三：初始化队列和片上缓冲区


本步骤为向量化运算配置运行所需硬件资源。输入队列负责接收从全局内存 GM 搬运而来的 gate、up 数据，输出队列用于临时存放最终运算结果，workBuf_缓冲区则承载负值、指数项、分母、激活值等各类中间计算数据。以上存储组件共同组成优化版本在片上完成运算的临时缓存工作区。


gateQueue_、upQueue_分别承担两路输入数据的搬运工作，yQueue_用于运算结果向全局内存回写，workBuf_统一存放全部中间张量。SwiGLU 的计算流程需要用到负 gate 值、指数运算结果、分母、常数 1、sigmoid、silu 等多组中间数据，因此程序按照 6 倍 tile 长度的总大小申请对应缓存空间，保障所有中间变量均可存放。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;"><strong>aicore</strong> inline void Init( GM_ADDR gate, GM_ADDR up, GM_ADDR y, const SwiGluTilingData &amp;tiling) { totalSize_ = tiling.totalSize; elementsPerCore_ = tiling.elementsPerCore; coreNum_ = tiling.coreNum; tileLength_ = tiling.tileLength; gateGm_.SetGlobalBuffer(reinterpret_cast&lt;<em><em>gm</em>_ float *&gt;(gate), totalSize_); upGm</em>.SetGlobalBuffer(reinterpret_cast&lt;<em><em>gm</em>_ float *&gt;(up), totalSize_); yGm</em>.SetGlobalBuffer(reinterpret_cast&lt;<em><em>gm</em>_ float *&gt;(y), totalSize_); const uint32_t tileBytes = tileLength</em> * sizeof(float); pipe_.InitBuffer(gateQueue_, 1, tileBytes); pipe_.InitBuffer(upQueue_, 1, tileBytes); pipe_.InitBuffer(yQueue_, 1, tileBytes); pipe_.InitBuffer(workBuf_, 6U * tileBytes); }</th>
</tr>
</thead>
</table>


## 步骤四：按tile组织搬运与计算


本步骤将各个运算核心负责的连续数据拆分为多个 tile。外层确定当前核心的整体处理范围，内层以 tileLength 为步长循环迭代，每次调用 ProcessTile () 完成单段数据的载入、运算与回写，实现大规模输入的分段流式执行。


Process () 依旧基于运算核心划分数据区间，内部不再逐元素运算，而是按照 tileLength 切分数据。每次调用 ProcessTile () 处理连续数据段，最后一个 tile 依靠 curLength 记录有效长度，适配长度无法被分块尺寸整除的输入场景。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;"><strong>aicore</strong> inline void Process() { const uint32_t coreId = GetBlockIdx(); if (coreId &gt;= coreNum_) { return; } const uint32_t start = coreId * elementsPerCore_; uint32_t end = start + elementsPerCore_; if (end &gt; totalSize_) { end = totalSize_; } for (uint32_t offset = start; offset &lt; end; offset += tileLength_) { uint32_t curLength = tileLength_; if (offset + curLength &gt; end) { curLength = end - offset; } ProcessTile(offset, curLength); } }</th>
</tr>
</thead>
</table>


## 步骤五：实现向量化SwiGLU公式


本步骤把SwiGLU公式拆成多个向量算子并串联起来：先计算负gate，再求指数和分母，随后得到sigmoid，再计算SiLU和最终输出。它展示了如何用向量接口表达一个融合算子，以及为什么流水中的每个阶段都需要依次完成。


优化版的核心在ProcessTile()。先用DataCopy搬入gate和up，再在片上缓冲区中完成sigmoid、silu和最终输出计算。每个向量操作后使用PipeBarrier<PIPE_V>()保证前后依赖顺序清晰。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">LocalTensor<float> workLocal = workBuf_.Get<float>(); LocalTensor<float> negGateLocal = workLocal; LocalTensor<float> expLocal = workLocal[tileLength_]; LocalTensor<float> denomLocal = workLocal[2U * tileLength_]; LocalTensor<float> oneLocal = workLocal[3U * tileLength_]; LocalTensor<float> sigmoidLocal = workLocal[4U * tileLength_]; LocalTensor<float> siluLocal = workLocal[5U * tileLength_]; Muls(negGateLocal, gateLocal, -1.0f, curLength); PipeBarrier<PIPE_V>(); Exp(expLocal, negGateLocal, curLength); PipeBarrier<PIPE_V>(); Adds(denomLocal, expLocal, 1.0f, curLength); PipeBarrier<PIPE_V>(); Duplicate(oneLocal, 1.0f, curLength); PipeBarrier<PIPE_V>(); Div(sigmoidLocal, oneLocal, denomLocal, curLength); PipeBarrier<PIPE_V>(); Mul(siluLocal, gateLocal, sigmoidLocal, curLength); PipeBarrier<PIPE_V>(); Mul(yLocal, siluLocal, upLocal, curLength); PipeBarrier<PIPE_V>();</th>
</tr>
</thead>
</table>


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">python3 tests/test_torch_op.py --rows 128 --hidden 1024 --atol 1e-3 --rtol 1e-3 ./out/bin/swiglu_optimized_standalone \ --rows 128 \ --hidden 1024 \ --block-dim 8 \ --tile-length 1024 \ --warmup 10 \ --repeat 50 \ --rounds 5</th>
</tr>
</thead>
</table>


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">python3 tests/compare_qwen_native.py \ --model /mnt/workspace/cann-learning-hub/contrib/tutorials/qwen_ops/Models/Qwen2.5-0.5B \ --repeat 3 \ --attn-implementation eager</th>
</tr>
</thead>
</table>


# 测试与验收


## 一、单算子正确性测试


该测试调用当前实验的PyTorch注册算子，并与同一数学语义的参考实现比较。终端输出全部为PASS（或ALL PASS）且进程返回码为0，表示正确性测试通过。


In [ ]:
%%bash
set -e

cd /mnt/workspace/cann-learning-hub/contrib/tutorials/qwen_ops
source ./setup_cannlab_env.sh
cd Qwen2.5cann_ops/SwiGluOptimizedExperiment
bash scripts/build.sh
python3 tests/test_torch_op.py


## 二、单算子执行时间测试


该指令先预热，再重复启动单个算子，并使用ACL Event统计设备侧执行时间。记录输出中的mean、median、min和max；该结果不包含Python参考计算、输入生成、结果比对及首次主机到设备的数据传输。基础版与优化版比较时，应使用相同输入形状、预热次数、重复次数和计算核心数。


In [ ]:
%%bash
set -e

cd /mnt/workspace/cann-learning-hub/contrib/tutorials/qwen_ops
source ./setup_cannlab_env.sh
cd Qwen2.5cann_ops/SwiGluOptimizedExperiment
export LD_LIBRARY_PATH="$PWD/out/lib:${LD_LIBRARY_PATH:-}"
./out/bin/swiglu_optimized_standalone --rows 128 --hidden 1024 --block-dim 8 --tile-length 1024 --warmup 10 --repeat 50 --rounds 5


## 任务拓展


完成基础实验后，可以继续围绕以下方向拓展：


• 在相同输入规模下对比基础版、优化版和原生算子的耗时。


• 调整任务划分和分块参数，观察正确性、吞吐和尾块处理是否变化。


• 将单算子测试、独立直调测试和模型替换测试的结果放在一起分析，区分算子内核内部耗时与端到端调度开销。


• 进一步尝试双缓冲、异步搬运、多级流水、FP16/BF16支持或更贴近硬件矩阵单元的实现。


## 实验总结


本实验展示了SwiGLU从基础版到优化版的核心改造：全局内存逐元素读写改为UB分块搬运，标量计算改为向量计算，并按搬入、计算、写回组织流程。优化版SwiGLU展示了把逐元素融合算子搬到片上缓冲区后，数据流会如何变化：输入先分块，随后在本地完成批量激活和乘法，最后按原形状写回。通过这个过程，可以把向量化和流水化理解为同一条数据链上的不同阶段。
